11- AUTOGLUON

In [7]:
# 📦 1. Importar librerías
import pandas as pd

In [9]:
# 💬 Instalar AutoGluon si es necesario
%pip install autogluon.timeseries

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

In [10]:
# 📄 2. Cargar datasets
df_sellin = pd.read_csv("C:/Users/Lenovo/Desktop/LABO 3/sell-in.txt", sep="\t")
df_productos = pd.read_csv("C:/Users/Lenovo/Desktop/LABO 3/tb_productos.txt", sep="\t")

In [11]:
# 📄 Leer lista de productos a predecir
with open("780_a_predecir.TXT", "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

In [12]:
# 🧹 3. Preprocesamiento
# Convertir periodo a datetime
df_sellin['timestamp'] = pd.to_datetime(df_sellin['periodo'], format='%Y%m')

In [13]:
# Filtrar hasta dic 2019 y productos requeridos
df_filtered = df_sellin[
    (df_sellin['timestamp'] <= '2019-12-01') &
    (df_sellin['product_id'].isin(product_ids))
]

In [14]:
# Agregar tn por periodo, cliente y producto
df_grouped = df_filtered.groupby(['timestamp', 'customer_id', 'product_id'], as_index=False)['tn'].sum()

In [15]:
# Agregar tn total por periodo y producto
df_monthly_product = df_grouped.groupby(['timestamp', 'product_id'], as_index=False)['tn'].sum()

In [16]:
# Agregar columna 'item_id' para AutoGluon
df_monthly_product['item_id'] = df_monthly_product['product_id']

In [17]:
# ⏰ 4. Crear TimeSeriesDataFrame
ts_data = TimeSeriesDataFrame.from_data_frame(
    df_monthly_product,
    id_column='item_id',
    timestamp_column='timestamp'
)

In [51]:
# Completar valores faltantes
ts_data = ts_data.fill_missing_values()

In [52]:
# ⚙️ 5. Definir y entrenar predictor
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS'
)

predictor.fit(ts_data, num_val_windows=2, time_limit=60*60,random_seed=14)

Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to 'C:\Users\Lenovo\Desktop\LABO 3\AutogluonModels\ag-20250719_203437'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          8
GPU Count:          0
Memory Avail:       0.55 GB / 7.91 GB (6.9%)
Disk Space Avail:   540.19 GB / 930.16 GB (58.1%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 14,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'time_limit': 3600,
 'verbosity': 2}

train_data with frequency 'IRREG' has been resampled to frequency 'MS'.
Provided train_data has 223

In [54]:
# 🔮 6. Generar predicción
forecast = predictor.predict(ts_data)

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


In [55]:
# Extraer predicción media y filtrar febrero 2020
forecast_mean = forecast['mean'].reset_index()
print(forecast_mean.columns)

Index(['item_id', 'timestamp', 'mean'], dtype='object')


In [56]:
# Tomar solo item_id y la predicción 'mean'
resultado = forecast['mean'].reset_index()[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']

# Filtrar solo febrero 2020
resultado = forecast['mean'].reset_index()
resultado = resultado[resultado['timestamp'] == '2020-02-01']

# Renombrar columnas
resultado = resultado[['item_id', 'mean']]
resultado.columns = ['product_id', 'tn']

In [72]:
# 💾 7. Guardar archivo
resultado.to_csv("submission_t780_autogluon_v1.csv", index=False)
resultado.head()

,product_id,tn
1,20001,1223.562637
3,20002,1023.347009
5,20003,683.813809
7,20004,520.523996
9,20005,487.932891


In [73]:
from pathlib import Path

# Ruta a tu carpeta de salidas
CARPETA_SALIDA = Path(r"C:\Users\Lenovo\Desktop\LABO 3")

# Nombre del archivo de salida
out_csv = CARPETA_SALIDA / "submission_t780_autogluon_v1.csv"

# 💾 7. Guardar archivo
resultado.to_csv(out_csv, index=False)

12- PROPHET

In [3]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from prophet import Prophet
from pandas.tseries.offsets import MonthBegin
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ───────── CONFIG ─────────
BASE_PATH      = Path(r"C:/Users/Lenovo/Desktop/LABO 3")
SALES_FILE     = BASE_PATH / "sell-in.txt"
LIST_FILE      = BASE_PATH / "780_a_predecir.txt"
OUTPUT_CSV     = BASE_PATH / "submission_prophet_feb2020.csv"
SEED           = 14
TARGET_DATE    = pd.Timestamp("2020-02-01")
FREQ           = "MS"     # frecuencia mensual
# ──────────────────────────

# reproducibilidad
random.seed(SEED)
np.random.seed(SEED)

# 1) Cargo y preparo datos
df = pd.read_csv(SALES_FILE, sep="\t", engine="python")
df.columns = df.columns.str.strip()
date_col = next(c for c in df.columns if "period" in c.lower())
df = df.rename(columns={date_col: "periodo"})
df["periodo"] = df["periodo"].astype(str).str.zfill(6)
df["ds"] = pd.to_datetime(df["periodo"] + "01", format="%Y%m%d", errors="coerce")

# 2) Lista de productos
with open(LIST_FILE, "r", encoding="utf-8") as f:
    product_ids = [int(line) for line in f if line.strip().isdigit()]

# 3) Para cada producto, entreno Prophet y predigo Feb 2020
results = []
for pid in tqdm(product_ids):
    # Serie mensual del target
    ts = (
        df[df["product_id"] == pid]
          .groupby("ds")["tn"]
          .sum()
          .asfreq(FREQ)
          .reset_index()
          .rename(columns={"tn": "y"})
    )
    # Si no hay datos, dejamos cero
    if ts["y"].isna().all():
        results.append({"product_id": pid, "tn": 0.0})
        continue

    # 4) Definir y entrenar modelo
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
        interval_width=0.95
    )
    m.fit(ts)

    # 5) Crear futuro hasta febrero 2020 y predecir
    periods = (TARGET_DATE.year - ts["ds"].dt.year.max()) * 12 + \
              (TARGET_DATE.month - ts["ds"].dt.month.max())
    future = m.make_future_dataframe(periods=periods, freq=FREQ)
    fcst = m.predict(future)

    # 6) Extraer pronóstico para 2020-02-01
    yhat_row = fcst.loc[fcst["ds"] == TARGET_DATE, "yhat"]
    pred = float(yhat_row.values[0]) if not yhat_row.empty else 0.0
    results.append({
        "product_id": pid,
        "tn": max(0, round(pred, 5))
    })

# 7) Guardar submission
sub_df = pd.DataFrame(results)
sub_df.to_csv(OUTPUT_CSV, index=False, float_format="%.5f")
print(f"✅ Submission Prophet para Febrero 2020 guardada → {OUTPUT_CSV.name}")

  0%|                                                                                          | 0/780 [00:00<?, ?it/s]10:55:13 - cmdstanpy - INFO - Chain [1] start processing
10:55:15 - cmdstanpy - INFO - Chain [1] done processing
  0%|                                                                                | 1/780 [00:06<1:28:27,  6.81s/it]10:55:16 - cmdstanpy - INFO - Chain [1] start processing
10:55:17 - cmdstanpy - INFO - Chain [1] done processing
  0%|▏                                                                                 | 2/780 [00:08<51:26,  3.97s/it]10:55:18 - cmdstanpy - INFO - Chain [1] start processing
10:55:19 - cmdstanpy - INFO - Chain [1] done processing
  0%|▎                                                                                 | 3/780 [00:11<44:40,  3.45s/it]10:55:21 - cmdstanpy - INFO - Chain [1] start processing
10:55:24 - cmdstanpy - INFO - Chain [1] done processing
  1%|▍                                                                  

✅ Submission Prophet para Febrero 2020 guardada → submission_prophet_feb2020.csv


13- AUTOGLUON MEJORADO

In [69]:
from pathlib import Path
from autogluon.timeseries import TimeSeriesPredictor

# ────────── RUTAS ──────────
BASE        = Path(r"C:\Users\Lenovo\Desktop\LABO 3")
OUTPUT_NAME = "submission_t780_autogluon_v2.csv"
# ──────────────────────────

# 1) Defino predictor sin tuning custom
predictor = TimeSeriesPredictor(
    target="tn",
    freq="MS",
    prediction_length=2,
    eval_metric="RMSE",
    path=str(BASE/"Autogluon_TSuper"),
    verbosity=2
)

# 2) Entreno empleando solo el preset “best_quality”
predictor.fit(
    train_data=ts_data,
    time_limit=3600,        # 1 hora en total
    presets="best_quality", # preset con ensembles y tuning interno
    enable_ensemble=True,   # activar ensemble final
    random_seed=14          # semilla reproducible
)

# 3) Predicción
forecast = predictor.predict(ts_data)

# 4) Extraigo y formateo febrero 2020
submission = (
    forecast["mean"]
      .reset_index()
      .query("timestamp=='2020-02-01'")
      .loc[:, ["item_id","mean"]]
      .rename(columns={"item_id":"product_id","mean":"tn"})
)

# 5) Guardo CSV
out_csv = BASE/OUTPUT_NAME
submission.to_csv(out_csv, index=False, float_format="%.5f")
print(f"✅ Submission guardada → {out_csv.name}")

Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to 'C:\Users\Lenovo\Desktop\LABO 3\Autogluon_TSuper'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          8
GPU Count:          0
Memory Avail:       1.25 GB / 7.91 GB (15.9%)
Disk Space Avail:   539.91 GB / 930.16 GB (58.0%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 14,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'time_limit': 3600,
 'verbosity': 2}

train_data with frequency 'IRREG' has been resampled to frequency 'MS'.
Provided t

✅ Submission guardada → submission_t780_autogluon_v2.csv
